<a href="https://colab.research.google.com/github/gpufit/Comet/blob/master/Colab_notebooks/COMET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![image](https://raw.githubusercontent.com/gpufit/Comet/master/Python_interface/resources/comet_logo_small.png)

#Cost-function Optimized Maximal overlap drift EsTimation

COMET is a software package designed to correct drift in single molecule localization microscopy (SMLM) datasets with optimal spatial and temporal resolution.

Check out the Comet [github repository](https://github.com/gpufit/Comet) for  detailed information.

Note: to try Comet with a simple test dataset, a sample dataset is availble [here](https://raw.githubusercontent.com/gpufit/Comet/master/test_dataset/test_dataset.csv).


# How to use the Colab notebook

## Step 1: Environment setup (run once)

1.   Ensure that you are logged into your Google account.
2.   Select from the dropdown menu: File -> Save a Copy in Drive.
3.   Select from the dropdown menu: Edit -> Notebook Settings and ensure that GPU hardware acceleration is selected.
4.   Run the next 2 blocks of code (below) only once, by clicking the "Play" buttons one at a time, to finish setting up the remote hardware and software environment.


In [ ]:
# @title 1.1 Install COMET
# numba-cuda is installed separately so it matches the CUDA driver Colab provides.
!uv pip install -q --system numba-cuda>=0.61.0
!pip install -q py-comet


In [ ]:
# @title 1.2 Import Numba
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1



---



## Step 2: Import a dataset
As a first step, you need to import an SMLM dataset that you want to drift correct.  

There are TWO options:

*   Option 1:   Load a dataset from your local PC.  To upload a dataset from a local drive, click Run for the FIRST code block below.
*   Option 2:   Load a dataset from Google drive (faster).  For this option, enter the link to the dataset in the text box, and click Run on the SECOND code block below.


In [ ]:
# @title 2.1 Option 1: Upload a dataset from a local drive
from google.colab import files
file = files.upload()
import numpy as np
import pandas as pd
for key in file.keys():
  filename = key
import io
data = pd.read_csv(io.BytesIO(file[filename]))

localizations = np.zeros((len(data['frame']), 4))
localizations[:, 0] = np.asarray(data['x [nm]'])
localizations[:, 1] = np.asarray(data['y [nm]'])
try:
  localizations[:, 2] = np.asarray(data['z [nm]'])
except:
  localizations[:, 2] = 0
localizations[:, 3] = np.asarray(data['frame'])
frames = np.unique(localizations[:, -1])
n_frames = len(frames)
print(f"{filename} import successful, {len(localizations[:, 0])} localizations, {n_frames} frames")

"""
#How to import other formats?
#If you use a different data format, modify the lines of code so that you end up
#with a numpy array called localizations that has the following dimensions:
#
#localizations.shape = (number_of_localizations, dataset_dimension+1), where
#localizations[:, 0] are the x-coordinates etc. and localizations[:, -1] are the
#corresponding frames.

from google.colab import files
file = files.upload()

#Your import code here

assert (len(localizations[0,:])==3 or len(localizations[0,:])==4)
frames = np.unique(localizations[:, -1])
n_frames = len(frames)
"""

In [ ]:
# @title 2.2 Option 2: Load a dataset stored on Google drive
filename = "test_dataset.csv" # @param {"type":"string"}

from google.colab import drive
drive.mount('/content/gdrive')
path_to_drive_files = "gdrive/MyDrive/"

import numpy as np
import pandas as pd

data = pd.read_csv(f"{path_to_drive_files}{filename}")

localizations = np.zeros((len(data['frame']), 4))
localizations[:, 0] = np.asarray(data['x [nm]'])
localizations[:, 1] = np.asarray(data['y [nm]'])
try:
  localizations[:, 2] = np.asarray(data['z [nm]'])
except:
  pass # just skip z localizations if not present
localizations[:, 3] = np.asarray(data['frame'])
frames = np.unique(localizations[:, -1])
n_frames = len(frames)
print(f"{filename} import successful, {len(localizations[:, 0])} localizations, {n_frames} frames")



---



## Step 3: Set data segmentation options

Set the Maximum drift parameter (**max_drift_nm**): this value will vary from microscope to microscope, and depends on environmental conditions.  The units of this varaible are **nanometers**.
* Choose an estimate (guess) of the maximum distance which the sample drifted during the acquisition.
* This number does not need to be exact, but it must be  larger than the true drift in the dataset.  
* If **max_drift_nm** is set too small, the output of COMET will exhibit artifacts (sharp jumps in the estimated drift).
* If **max_drift_nm** is set too large, COMET may take a very long time to run, or may report a memory error.
* Recommended setting: for typical SMLM experiments, max_drift = 300nm - 2000nm.

Next, choose how the dataset will be segmented into time windows.  The larger the number of windows, the higher the time resolution of the drift estimate.

There are three options to segment the localization data:

* Option 1: **Choose the number of segments (N_seg):** divide the dataset into **N_seg** parts, each containing an equal number of localizations.
* Option 2: **Choose the number of localizations per segment (N_loc):** divide the dataset into equal parts, each containing **N_loc** localizations.
* Option 3: **Choose the number of camera frames per time window (N_frm):** divide the dataset into equal-length time segments, with **N_frm** per segment.  Note that if a segment contains zero localizations, the algorithm will fail.

Depending on which option is chosen, the **segmentation_parameter** corresponds to either **N_seg**, **N_loc**, or **N_frm**.

After running the segmentation code, the number of localization pairs (**N_pairs**) will be calculated.  If this number is too high for the CPU/GPU, a warning message is displayed.  In this case, the **downsampling slider** may be used to downsample the data, and the segmentation code can be run again to reduce **N_pairs** to a value which is compatible with the computer.  For Google Colab use, this number should be smaller than 250 x 10^6.


In [ ]:
# @title 3.1 Run data segmentation

import numpy as np
from typing import Optional, Dict
from dataclasses import dataclass
from comet.core.segmenter import segmentation_wrapper

def estimate_pairs(coordinates):
  for i in range(len(coordinates[0])):
    coordinates[:, i] -= np.min(coordinates[:, i])
  coordinates = np.array(np.floor(coordinates / max_drift_nm), dtype=int)
  coordinates = np.array(list(map(tuple, coordinates)))
  sort_indices = np.lexsort(coordinates.T)# get the unique tuples and their counts
  unique_tuples, counts = np.unique(coordinates[sort_indices], axis=0, return_counts=True)
  # get the indices of the similar tuples
  similar_indices = np.split(sort_indices, np.cumsum(counts[:-1]))
  idx_i = []
  idx_j = []
  pair_idc_estimate = 0
  for i in range(len(similar_indices)):
    n_elements = len(similar_indices[i])
    pair_idc_estimate += n_elements * (n_elements - 1)
  rounded = round(pair_idc_estimate,-4)
  print(f"{int(rounded):,d} pairs.")
  if rounded > 100000000:
    print("This mustn't exceed 250 million! Otherwise use the downsampling slider to reduce ")
  elif rounded < 100000:
    print("Usually in the range of millions, tens of millions up to 100 million. Potentially try increasing the max drift.")
#@title 4.1 Set Maximum Drift
max_drift_nm = 200#@param {type:"number"}
max_drift_nm = float(max_drift_nm)

# Example inputs (replace these with your actual vars in Colab):
segmentation_method = "segment by number of frames per window" # @param ["segment by number of time windows","segments by number of locs per window","segment by number of frames per window"]
segmentation_parameter = 50                                # @param {type:"integer"}

downsampling = 1.0  # @param {type:"slider", min:0.025, max:1, step:0.025}

# STEP A: pull out the 1‐D array of frame‐indices (must be ints)
loc_frames = localizations[:, -1].astype(int)

# STEP B: map the old‐style string → “mode” integer for the new API
mode_map = {
    "segment by number of time windows":          0,   # → segment_by_num_windows
    "segments by number of locs per window":      1,   # → segment_by_num_locs_per_window
    "segment by number of frames per window":     2,   # → segment_by_frame_windows
}
if segmentation_method not in mode_map:
    raise ValueError(f"No such segmentation method: {segmentation_method!r}")
min_frame = np.min(loc_frames)
max_frame = np.max(loc_frames)
seg_mode = mode_map[segmentation_method]
if seg_mode == 0:
  mean_locs_per_frame = len(loc_frames) // segmentation_parameter
  max_locs_per_segment = int(downsampling * mean_locs_per_frame)
elif seg_mode == 1:
  max_locs_per_segment = int(downsampling * segmentation_parameter)
else:
  mean_locs_per_frame = len(loc_frames) * segmentation_parameter / len(np.unique(loc_frames))
  max_locs_per_segment = int(downsampling * mean_locs_per_frame)
print(f"Using on average {max_locs_per_segment} locs per time window will lead to approximately:")
seg_result = segmentation_wrapper(loc_frames, segmentation_parameter,
                                  mode_map[segmentation_method],
                                  max_locs_per_segment=max_locs_per_segment,
                                  return_param_dict=True)
estimate_pairs(localizations[seg_result.loc_valid, :-1].copy())



---



## Step 4: Run the COMET algorithm

Run the next code block to use COMET to estimate the drift of your segmented localizations.

**target_sigma_nm** is the final target Gaussian length scale for fine refinement. The smaller this value, the more precise the correction becomes but the longer the analysis takes. Default: 10 nm

In [ ]:
# @title 4.1 Run COMET
target_sigma_nm = 10 # @param {"type":"number"}

from comet.core.drift_optimizer import comet_run_kd
import time
import warnings
t = time.time()
warnings.filterwarnings("ignore")

drift = comet_run_kd(dataset=localizations, 
                                  segmentation_mode=mode_map[segmentation_method],
                                  segmentation_var=segmentation_parameter,
                                  max_locs_per_segment=max_locs_per_segment,
                                  target_sigma_nm=target_sigma_nm,
                                  max_drift_nm=max_drift_nm)

print(f"algortihm done in {np.round(time.time()-t)}s")



---



## Step 5: Display and download the results

The COMET algorithm is complete!  Below are the options to inspect the results and download them for later use.

In [ ]:
#@title 5.1 Plot COMET drift estimate
import matplotlib.pyplot as plt

plt.plot(drift[:, -1], drift[:, 0]-drift[:,0].mean())
plt.plot(drift[:, -1], drift[:, 1]-drift[:,1].mean())
plt.plot(drift[:, -1], drift[:, 2]-drift[:,2].mean())
plt.xlabel("Frames")
plt.ylabel("Drift estimate [nm]")
plt.legend(["x est.", "y est.", "z est."]);

In [ ]:
# @title 5.2 Save the COMET drift estimate as CSV
from google.colab import files
import pandas as pd

result = np.zeros((len(drift[:, 0]), 4))
result[:, 1] = drift[:, 0]
result[:, 2] = drift[:, 1]
result[:, 3] = drift[:, 2]
result[:, 0] = drift[:, -1]
header = "frame,x_nm,y_nm,z_nm\n"

df = pd.DataFrame(result, columns=["frame", "x_nm", "y_nm", "z_nm"])
df.to_csv("drift_trajectory_result.csv", index=False, sep=",", float_format = '%.3f')
files.download('drift_trajectory_result.csv')


In [ ]:
# @title 5.3 Download drift-corrected SMLM dataset as CSV
from google.colab import files
import csv

dataset_corrected = np.zeros_like(localizations)
# Correct and reorder the dataset to be frame,x,y,z
for i in range(3):
  dataset_corrected[:, i+1] = localizations[:, i] - drift[localizations[:, -1].astype(int), i]
dataset_corrected[:, 0] = localizations[:, -1]

corrected_df  = data.copy(deep=True)
corrected_df["frame"] = dataset_corrected[:, 0]
corrected_df["x [nm]"] = dataset_corrected[:, 1]
corrected_df["y [nm]"] = dataset_corrected[:, 2]


header = ['"frame"', '"x [nm]"', '"y [nm]"']
# Add z coordinates if present
if "z [nm]" in corrected_df.columns:
    corrected_df["z [nm]"] = dataset_corrected[:, 3]
    header.append('"z [nm]"')

savename = 'corrected_dataset.csv'
header = ['"'+col+'"' for col in corrected_df.columns.to_list()]
corrected_df.to_csv(savename, index=True,
          header=header, index_label='"id"',
          mode='w', quoting=csv.QUOTE_NONE,
          float_format = '%.3f') # precision of 3

files.download(savename)